# Titel der Analyse

Untertitel

Dein Name

## LLM-Setup

Konfiguriere deinen AI-Helfer einmalig vor Beginn deiner Analyse. Wenn etwas schief geht, kannst du die folgenden Zellen nochmals ausführen, um in den Startzustand zurückzukehren.

Entscheide dich, ob du **entweder** Ollama-Cloud **oder** einen eigenen lokalen Ollama-Server verwenden willst.

### Ollama Cloud

Dein Computer ist zu langsam oder hat nicht genügend RAM (16GB+)? Dann verwende Ollama Cloud. Informationen zu Ollama-Cloud findest du unter [docs.ollama.com/cloud](https://docs.ollama.com/cloud#cloud-models)

In [ ]:
OLLAMA_API_KEY="243tg524564tzg24c549ecb32a9ed7e0446.8in8Dd5NcFYhzLObRIb_uz0g"

Deinen eigenen privaten API-Schlüssel erstellst du nach dem Login unter [ollama.com/settings/keys](https://ollama.com/settings/keys).

In [ ]:
import ollama

client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

### Ollama Local

**Führe die folgende Zellen nur aus, wenn du Ollama Cloud nicht verwendest.**

Alternativ kannst du Ollama auch lokal laufen lassen.

In [ ]:
client = ollama.Client(
#  host='http://127.0.0.1:11434' # Ollama und Juypter laufen lokal
  host='http://10.0.2.2:11434' # Ollama äuft lokal und Jupyter in der VM
)

Ob unter einer bestimmten (lokalen) IP-Adresse und Port ein Ollama-Server läuft, kannst du leicht so testen:

In [ ]:
%%bash
#curl --silent http://127.0.0.1:11434
curl --silent http://10.0.2.2:11434

### Hilfsfunktionen

Für die einfachere Handhabung nutzen wir die zwei selbstgeschriebenen Funktionen `ask()` und `chat()`. Unser Code im Notebook ist dann übersichtlicher. Führe diese Code-Zellen bei jeder Verwendung dieses Notebooks einmalig aus.

In [ ]:
from IPython.display import display, Markdown

def ask(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell - ohne Historie"""
    global client
    global model_name
    response = client.generate(model=model_name, prompt=prompt)
    display(Markdown(response.response))

def remove_message(msg_list, n=1):
    #print("Cleanup - removing {} oldest message from chat history".format(n))
    return msg_list[0:1] + msg_list[(n+1):]
    
def chat(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell und Historie"""
    global client
    global model_name
    global messages
    global max_chat_history
    
    messages.append({"role": "user", "content": prompt})

    if max_chat_history != 0:
        message_count = len(messages) - 1
        
        # remove exess chat history, if we have fixed size max_chat_history
        excess_messages = message_count - max_chat_history
        if max_chat_history > 0 and excess_messages > 0:
            messages = remove_message(messages, excess_messages)

        # if context window is (still) to long, we always remove old chat entries
        for _ in range(message_count + 1):
            try:
                response = client.chat(model=model_name, messages=messages)
            except ollama.ResponseError as e:
                if len(messages) > 1:
                    messages = remove_message(messages, 1)
                    continue
                raise
            except Exception as e:
                raise
    else:
        # no context lenght limit - expect your client to die eventually
        response = client.chat(model=model_name, messages=messages)
        
    messages.append({"role": "assistant", "content": response.message.content})
    display(Markdown(response.message.content))

def show_models_from_(output):
    """Zeigt die Modelle in einer ollama-python-client-Ausgabe an"""
    for entry in output['models']:
        print(entry["model"])

Wichtig: Die globalen Variablen `model_name` und `messages` und `max_chat_history` müssen vor der Verwendung definiert werden. Dazu die folgenden Code-Zellen einmalig nacheinander ausführen. 

### Chat-History


Die Funktion `chat()` verwendet auch die bisherige Konversation als Eingabe - so "erinnert" sich ein Modell, was alles schon besprochen wurde. Die maximal Länge der Eingabe ist von Modell zu Modell verschieden aber immer begrenzt. Mit der globalen Variable `max_chat_history` kannst du einstellen, wie viele der vorhergehenden Fragen und Antworten im `chat()` verwendet werden.

|Wert|Bedeutung|
|--|--|
|-1|Model Limit - das heisst, dass dein Chat-Client soviel History wie möglich verwendet. Wenn du einen langsamen Computer hast, ist diese Einstellung keine gute Idee. Wenn du Ollama-Cloud verwendest oder einen sehr schnellen Computer hast, dann ist ggf. -1 eine gute Wahl|
|0|ohne Limit - das heisst, dass dein Chat-Client irgendwann abstützen wird - gut, zum Testen der Funktionalität!|
|3,4,5,...|festes Limit - Wenn du bspw. 3 wählst, dann "erinnert" sich dein Chatbot maximal an die drei vorhergehenden Fragen und Antworten. Wenn Ollama lokal läuft und/oder dein Computer langsam ist, solltest du eine kleine Zahl wie bspw. `5` wählen. Versuch macht kluch!|

In [ ]:
max_chat_history = 5

### Modellauswahl

Modellübersicht: https://ollama.com/search

Welche Modelle gibt es auf deinem Ollama-Server oder bei Ollama-Cloud?

In [ ]:
show_models_from_(client.list())

Wenn Ollama lokal läuft, achte auf die Grösse des Modells (in GB) im Verhältnis zum verfügbaren Arbeitsspeicher. Grösser heisst i.d.R. auch langsamer - aber nicht zwangsläufig auch immer besser.

Mit welchem Modell willst du arbeiten?

In [ ]:
model_name = 'gemma3:4b' # Starte mit diesem, wenn du Ollama-Cloud verwendest

#model_name = 'gemma3:270m' # Starte mit diesem, wenn du Ollama lokal laufen lässt
#model_name = 'gemma3:1b'

# weitere Beispiele
#model_name = 'deepseek-r1:1.5b'
#model_name = 'codegemma:2b'

Wenn du nicht Ollama-Cloud verwendest, installierst du neue Modelle auf deinem Ollama-Server so:

In [ ]:
#client.pull(model_name)

### Priming

Wir speichern die gesamte Unterhaltung in der globalen Variablen `messages`. Am Anfang können wir das Modell noch `primen`.

In [ ]:
priming = """
You are the best Python coder in the world.
You know Pandas and all of Pandas functionality inside out.
You use Pandas data frames for plotting and avoid using matplotlib directly.
You code like a student in 10th grade and prefer simple solutions.
You give short and informative answers.
"""

In [ ]:
messages = [
    {"role": "system", "content": priming}
]

### Modelltest

Für einmalige Fragen nutzen wir `ask()`

In [ ]:
ask("Erzähle mir einen Informatik-Lehrerwitz")

Für Unterhaltungen nutzen wir `chat()` - mit `"""` auch mehrzeilig.

In [ ]:
chat("""
Ich benötige Hilfe bei der Datenanlyse mit Python und Pandas.
Wie gehe ich von den Rohdaten bis zur Visualisierung meiner Ergebnisse Schritt für Schritt vor?
""")

## Forschungsfragen

1. ...
2. ...
3. ...

## Daten einlesen

In [ ]:
chat("""
Wie liest man eine komma-separierte Datei in Python mit Pandas ein, 
wenn das Trennzeichen ein Semikolon ist?
""")

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('pfad/zur/csv/datei.csv', sep=";")
df

## Daten vorverarbeiten

## Daten analysieren

## Daten visualisieren